# Citation graph of the selected papers

This notebook retrieves **which selected paper cites which other selected paper** using the Semantic Scholar Academic Graph API. A directed edge **A → B means A cites B**. This is a bibliographic graph, independent of embedding similarity and cluster assignments.

The selection comes from the same section-based embedding manifest and exclusion file as the clustering notebooks. Original `Pxxx` labels are preserved; PDF SHA-256 hashes are the stable node keys. Excluded papers do not enter the graph. No PDF is uploaded, and no embedding is recomputed: DOI/title queries and paper identifiers are sent to Semantic Scholar.

This notebook contains the complete workflow for the thesis submission: selection, API authentication, candidate matching, recorded identity review, reference retrieval, graph export and inspection. The implementation is included in `src/batill/`; the reviewed decisions are in `configs/citation_match_review.json`.

**Current run:** matching and review are already complete. Run all sections in order with the **ml** kernel: section 3 reuses a complete saved matching table by default, section 3b applies the recorded review, and section 4 resumes reference retrieval from the API cache. Set `RERUN_MATCHING = True` only to recompute proposals or apply changes to matching overrides. Keep `REFRESH_API_CACHE = False` to reuse existing responses.

**Short resume path:** after a kernel restart, run sections **1 → 2 → 3b → 4 → 5**. This skips only the already-completed matching stage; its code remains documented below. Section 6 is optional inspection. No archived notebook is required to run or understand the workflow.

The restored notebook cells have not been rerun by the assistant. Previous execution snapshots remain in the recovery archive.

**Method:** resolve DOI/title identities → review uncertain matches → paginate outgoing references → retain links with both ends in the selection → export graph and coverage tables. Unmatched papers remain as nodes with explicit missing-data status. An isolated node is not evidence that a paper has no citations.

API documentation: [Academic Graph API](https://api.semanticscholar.org/api-docs/graph) and [official tutorial](https://www.semanticscholar.org/product/api/tutorial).

## 1. Load the current selection and bibliographic metadata

Use the **ml** kernel. If dependencies are missing, run `python -m pip install -e '.[citations,notebook]'` from the project directory in that environment.

Filenames and exact normalized bibliography-title matches supply better search titles and DOIs where possible. These are matching aids: they do not change the original PDFs or embedding metadata. Conflicting bibliography DOIs are left blank. Inspect the metadata table if a query title looks wrong.

Outputs are separated by a hash of the selected paper IDs. Changing the exclusion file and rerunning creates a new selection folder when membership changes. The API response cache is shared so previous lookups can be reused.

In [ ]:
from pathlib import Path
import sys

# Find the source package before importing project modules.
ROOT = next((p for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             for p in (parent, parent / 'batill')
             if (p / 'src' / 'batill').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the batill project folder.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Python:', sys.executable)

import pandas as pd
from IPython.display import display
from batill.citation_graph import (
    citation_selection, resolve_papers, retrieve_references, export_citation_graph,
)
from batill.semantic_scholar import ScholarClient
from batill.storage import write_json

CORPUS_DIR = ROOT / 'outputs' / 'corpus_sections'
EXCLUSIONS = ROOT / 'configs' / 'analysis_exclusions.json'
BIBLIOGRAPHIES = sorted((ROOT / 'data' / 'bibliography').glob('*.csv'))

papers, exclusion_audit, selection = citation_selection(
    CORPUS_DIR, EXCLUSIONS, BIBLIOGRAPHIES,
)
BASE = ROOT / 'outputs' / 'citation_graph'
RUN_DIR = BASE / selection['selection_id'][:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)
papers.to_csv(RUN_DIR / 'selected_papers.csv', index=False)
exclusion_audit.to_csv(RUN_DIR / 'exclusion_audit.csv', index=False)
write_json(RUN_DIR / 'selection.json', selection)
print(f"Selected papers: {len(papers)}; with bibliography DOI: {papers.doi.ne('').sum()}")
print('Results:', RUN_DIR)
display(papers[['paper_id', 'query_title', 'query_year', 'doi', 'metadata_status']])

## 2. Enter the API key privately

Enter your key at the hidden prompt; **do not paste it into a code cell**. Alternatively, define the `SEMANTIC_SCHOLAR_API_KEY` environment variable before starting the kernel. The client holds the key in memory and does not write it to its cache or result files.

Requests start at one every two seconds. After a rate limit, the client waits 20 seconds before each retry (longer only if the server requests it through Retry-After), and temporarily slows later requests. After every three successful live responses, it halves the interval back towards the normal two seconds; cached responses do not count as API recovery. Retry messages show that the notebook is waiting, not stuck. A first corpus run can take several minutes. Each successful API response is cached; after an interruption, rerun the stage to reuse completed requests. Keep `REFRESH_API_CACHE=False` for resuming. Set it to `True` only when deliberately refreshing provider data; this overwrites cached responses for requested endpoints.

In [ ]:
import os
from getpass import getpass

REFRESH_API_CACHE = False
api_key = os.environ.get('SEMANTIC_SCHOLAR_API_KEY') or getpass('Semantic Scholar API key: ')
client = ScholarClient(api_key.strip(), BASE / 'api_cache', refresh=REFRESH_API_CACHE)
del api_key

## 3. Resolve paper identities and review matches

Automatic acceptance requires an exact normalized title (at least four words) and either a matching DOI or matching year with no close competing search candidate and no conflicting DOI. This is a conservative heuristic, not a measured matching-accuracy guarantee. Check a sample of automatically accepted matches too. The title-similarity score is a string comparison, not a probability.

Uncertain proposals remain **unapproved** and will not contribute citation links. Review their Semantic Scholar titles, years, authors and URLs in `match_candidates.csv`. To correct a match, add an explicit Semantic Scholar paper ID or `DOI:...` to `MATCH_OVERRIDES`. To accept a reviewed top proposal, add its local ID to `APPROVE_PAPERS`. Use `None` as an override to leave a paper deliberately unmatched. Rerun this cell and the subsequent stages after any edits. Do not edit the generated CSVs: rerunning replaces them.

If two selected PDFs resolve to the same Semantic Scholar identity, both are flagged for review. Resolve the duplication explicitly rather than representing one indexed paper as two graph nodes with the same identity.

**Reuse versus recompute:** `RERUN_MATCHING = False` loads a saved table if it completely covers the current selection and its query titles/DOIs still agree. Missing or incomplete tables trigger the matching code below, which resumes from cached API responses. Set the flag to `True` after changing `MATCH_OVERRIDES`, `APPROVE_PAPERS` or the acceptance setting. Recorded manual decisions in section 3b remain separate and reject stale proposals.

In [ ]:
AUTO_ACCEPT_STRICT_MATCHES = True
MATCH_OVERRIDES = {
    # 'P001': 'DOI:10.xxxx/example',  # Replace with a verified identifier.
    # 'P002': None,                 # Deliberately leave this paper unmatched.
}
APPROVE_PAPERS = []  # For example ['P003'], only after reviewing its proposed match.

RERUN_MATCHING = False
saved_path = RUN_DIR / 'matches.csv'
saved_matches = pd.read_csv(saved_path).fillna('') if saved_path.exists() else None
reuse_matches = (
    not RERUN_MATCHING
    and saved_matches is not None
    and saved_matches.Key.tolist() == papers.Key.tolist()
    and saved_matches.query_title.tolist() == papers.query_title.tolist()
    and saved_matches.doi.tolist() == papers.doi.tolist()
)
if reuse_matches:
    matches = saved_matches
    print('Loaded existing matching results; no matching API requests made.')
else:
    write_json(RUN_DIR / 'matching_settings.json', {
        'auto_accept_strict_matches': AUTO_ACCEPT_STRICT_MATCHES,
        'overrides': MATCH_OVERRIDES, 'approved_proposals': APPROVE_PAPERS,
    })
    matches = resolve_papers(
        client, papers, RUN_DIR, overrides=MATCH_OVERRIDES,
        approve_papers=APPROVE_PAPERS, auto_accept=AUTO_ACCEPT_STRICT_MATCHES,
    )

display(matches.groupby(['match_status', 'approved']).size().rename('papers').to_frame())
review_columns = ['paper_id', 'query_title', 'query_year', 's2_title', 's2_year',
                  's2_id', 'title_similarity', 'match_status', 's2_url']
display(matches.loc[~matches.approved, review_columns])
print('Approved:', int(matches.approved.sum()), '/', len(matches))
print('Review files:', RUN_DIR / 'matches.csv', RUN_DIR / 'match_candidates.csv')

In [ ]:
# Inspect one uncertain paper and its search candidates; change PAPER_TO_REVIEW.
uncertain_ids = matches.loc[~matches.approved, 'paper_id'].tolist()
PAPER_TO_REVIEW = uncertain_ids[0] if uncertain_ids else papers.iloc[0].paper_id
try:
    candidates = pd.read_csv(RUN_DIR / 'match_candidates.csv').fillna('')
except pd.errors.EmptyDataError:
    candidates = pd.DataFrame()
display(papers.loc[papers.paper_id.eq(PAPER_TO_REVIEW)])
if not candidates.empty:
    display(candidates.loc[candidates.paper_id.eq(PAPER_TO_REVIEW)])

## 3b. Apply the recorded identity review (no API calls)

The assisted review checked the **110 initially unapproved papers** against PDF first pages, local bibliography records and cached search candidates. It approved 97, including nine candidate replacements; 13 records remain pending. The original 184 automatic matches were not manually re-reviewed.

The decisions are saved by PDF hash in `configs/citation_match_review.json`, with reasons and pinned candidate metadata. Under the agreed policy, verified versions of the same work can be used, but `same_work_version_not_verified` marks that their reference lists may differ from the PDFs. Matching-DOI publication identities have a separate status. The cell writes `reviewed_matches.csv` and an audit; it preserves the original `matches.csv`.

**After the initial matching run, you can run this cell directly without rerunning section 3.** Setup must have defined `papers` and `RUN_DIR`. Then continue to section 4 using the reviewed `matches`. If you restarted the kernel, run setup and key entry first. Changing a reviewed paper's proposed identity, DOI or query title makes its decision stale and stops application; update or remove that record in the review JSON before applying a new decision.

The two duplicate PDF pairs remain unapproved until the selection is resolved. No PDFs, exclusion decisions or embeddings have been changed. P204 is correctly identified as an erratum; its scope status is for you to decide.

See `review/report.md` for findings and `review/pending_review.csv` for the remaining cases.

In [ ]:
from batill.citation_review import apply_match_review

REVIEW_FILE = ROOT / 'configs' / 'citation_match_review.json'
raw_matches = pd.read_csv(RUN_DIR / 'matches.csv').fillna('')
if raw_matches.Key.tolist() != papers.Key.tolist():
    raise ValueError('Saved matches do not cover the current selection in order.')
matches, review_audit = apply_match_review(raw_matches, REVIEW_FILE)
(RUN_DIR / 'review').mkdir(parents=True, exist_ok=True)
matches.to_csv(RUN_DIR / 'reviewed_matches.csv', index=False)
review_audit.to_csv(RUN_DIR / 'review' / 'review_decisions.csv', index=False)
review_audit.loc[review_audit.decision.eq('hold')].to_csv(
    RUN_DIR / 'review' / 'pending_review.csv', index=False,
)
print(f'Approved for reference retrieval: {int(matches.approved.sum())}/{len(matches)}')
display(matches.groupby('match_status').size().rename('papers').to_frame())
display(review_audit.loc[review_audit.decision.eq('hold'),
                        ['paper_id', 'query_title', 'category', 'reason', 'suggested_lookup']])

## 4. Retrieve outgoing references

Run this after inspecting the matches. Only approved identities are queried; it is also possible to make a partial graph while some matches still need review. Every page of each approved paper's outgoing reference list is requested. Incoming citation queries are unnecessary for the induced graph: every internal edge should appear in the citing paper's reference list **if Semantic Scholar indexes that reference correctly**.

`complete_api_pagination` means that all API pages were read, not that the publisher's bibliography is fully indexed. Null reference IDs, retrieval errors and differences from the provider's reported reference count are recorded. Papers with no returned references require caution, especially if other coverage indicators look incomplete.

In [ ]:
if not matches.attrs.get('review_config_id'):
    raise ValueError('Run section 3b to apply the recorded identity review first.')
if not matches.approved.any():
    raise ValueError('No approved identities. Review and correct matches in section 3 first.')
print(f'Retrieving references for {int(matches.approved.sum())} approved papers; '
      f'{int((~matches.approved).sum())} remain unmatched/unapproved.')
retrieval, all_references = retrieve_references(client, papers, matches, RUN_DIR)
display(retrieval)
display(retrieval.loc[
    retrieval.status.ne('complete_api_pagination')
    | retrieval.unresolved_reference_ids.gt(0)
    | retrieval.reported_count_differs
])

## 5. Export the directed graph and inspect coverage

The graph includes all selected papers and only observed, non-self citation links whose two identities were approved. Duplicate API reference records become one binary edge. References to papers outside the selection are retained in the audit table but excluded from the graph.

- `nodes.csv`: paper identities, matching/retrieval status and within-selection citation counts.
- `edges.csv`: citing → cited paper IDs and stable PDF hashes.
- `citation_graph.graphml`: directed graph for NetworkX, Gephi or other graph software.
- `adjacency.npz` and `adjacency_order.csv`: sparse matrix with **row = citing paper, column = cited paper**.
- `matches.csv`, `match_candidates.csv`, `retrieval_status.csv`, `all_resolved_references.csv`: evidence and review tables.
- `run_manifest.json`: selection, retrieval timestamps, cache/output hashes and software versions.

Treat this as an observed Semantic Scholar graph. Missing matches or reference IDs can remove real edges; they do not justify excluding a paper from the thesis. If the final selection changes, rerun from section 1. Previously stored API responses can be reused.

`nodes.csv` and GraphML retain the version-review status. The summary counts version proxies; these are approved scholarly-work matches, not a claim of identical editions. Keep this limitation when comparing the graph with references extracted from the local PDFs.

To link references across publication versions, export first matches the approved Semantic Scholar ID and then uses an exact DOI alias only when it identifies one approved selected paper. `target_match_method` records that choice in `edges.csv`. References whose ID and DOI identify different selected papers are excluded and listed in `reference_identity_conflicts.csv`. There is no fuzzy title matching at this stage.

In [ ]:
nodes, edges, graph, graph_summary = export_citation_graph(
    papers, matches, retrieval, all_references, RUN_DIR, selection, client,
)
display(pd.Series(graph_summary, name='value').to_frame())
display(edges.head(30))
print('Graph:', RUN_DIR / 'citation_graph.graphml')
print('All result files:', RUN_DIR)

## 6. Inspect citation links for a selected paper

The counts below refer only to citations **within this selection**, not total scholarly impact. Incoming and outgoing links answer different questions: which selected papers cite this one, and which selected papers this one cites?

In [ ]:
PAPER_TO_INSPECT = papers.iloc[0].paper_id  # Change to an original Pxxx ID.
lookup = nodes.set_index('Key')
selected_node = nodes.loc[nodes.paper_id.eq(PAPER_TO_INSPECT)]
if selected_node.empty:
    raise ValueError('This paper ID is not in the current selection.')
key = selected_node.iloc[0].Key
display(selected_node[['paper_id', 'query_title', 'approved', 'status']])
print('This paper cites:')
display(lookup.loc[list(graph.successors(key)), ['paper_id', 'query_title']])
print('This paper is cited by:')
display(lookup.loc[list(graph.predecessors(key)), ['paper_id', 'query_title']])
print('Most cited papers within the observed selection:')
display(nodes.sort_values('citations_received_within_selection', ascending=False)[
    ['paper_id', 'query_title', 'citations_received_within_selection',
     'references_within_selection', 'match_status', 'status']
].head(20))